In [1]:
from langchain_community.document_loaders import TextLoader, DataFrameLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
# print("All imports successful!")

c:\Users\goyal\OneDrive\Desktop\Book-Recommendation-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("All imports Pretty  Successfuly !")

All imports Pretty  Successfuly !


In [4]:
from sentence_transformers import SentenceTransformer

# This will download the model (approx. 80MB) on the first run
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = ["This is an example query", "Each sentence is converted into a vector"]
embeddings = model.encode(sentences)

print(f"Embedding shape: {embeddings.shape}") # Should be (2, 384)

Embedding shape: (2, 384)


In [ ]:
# import os
# from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# # 1. Load your API key
# load_dotenv()
# # In your setup code
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3-pro-preview",    
#     google_api_key=os.getenv("GOOGLE_API_KEY"),
#     temperature=0.0
# )
# # AND CHANGE YOUR EMBEDDINGS TO:
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001",  # ✅ Confirmed working embedding model
#     # google_api_key=os.getenv("GOOGLE_API_KEY")                # ✅ Don't forget to pass the key here too!
# )

In [3]:
import pandas as pd 

books=pd.read_csv("books_cleaned.csv")

books["tagged_description"].to_csv("tagged_descriptions.text",
                                    sep="\n" , index=False, header=False)

In [4]:
# 1. Ensure TextLoader is imported (Fixes the NameError from before)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_data = TextLoader("tagged_descriptions.text", encoding="utf-8").load()

# 2. Set chunk_size to 1000 (roughly the length of a book summary)
# and chunk_overlap to 100 (keeps context between chunks)
text_splitter = CharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100, 
    separator="\n"
)

# 3. This will now run without the ValueError
documents = text_splitter.split_documents(raw_data)



Created a chunk of size 1170, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1090, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1269, which is longer than the specified 1000
Created a chunk of size 2012, which is longer than the specified 1000
Created a chunk of size 1227, which is longer than the specified 1000
Created a chunk of size 1186, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1193, which is longer than the specified 1000
Created a chunk of size 1059, which is longer than the specified 1000
Created a chunk of size 1272, which is longer than the specified 1000
Created a chunk of size 1637, which is longer than the specified 1000
Created a chunk of size 1134, which is longer than the specified 1000
Created a chunk of s

Created a chunk of size 1006, which is longer than the specified 1000
Created a chunk of size 2512, which is longer than the specified 1000
Created a chunk of size 1424, which is longer than the specified 1000
Created a chunk of size 1085, which is longer than the specified 1000
Created a chunk of size 1146, which is longer than the specified 1000
Created a chunk of size 1270, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of size 1814, which is longer than the specified 1000
Created a chunk of size 1832, which is longer than the specified 1000
Created a chunk of size 1066, which is longer than the specified 1000
Created a chunk of size 1182, which is longer than the specified 1000
Created a chunk of size 1286, which is longer than the specified 1000
Created a chunk of size 1644, which is longer than the specified 1000
Created a chunk of size 1932, which is longer than the specified 1000
Created a chunk of s

In [14]:
import os
import shutil
import time
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. SETUP MODEL (This is your all-MiniLM-L6-v2)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})

# 2. PATH SETUP
db_path = "./book_db_final_version" 
if os.path.exists(db_path):
    shutil.rmtree(db_path)

# 3. INITIAL BATCH (Creates the database)
print("🚀 Starting ingestion...")
batch_size = 40
db_books = Chroma.from_documents(
    documents=documents[:batch_size], 
    embedding=hf_embeddings, 
    persist_directory=db_path
)
print(f"✅ Initial {batch_size} books stored.")

# 4. LOOP FOR REMAINING BOOKS
for i in range(batch_size, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    db_books.add_documents(batch)
    print(f"✅ Progress: {i + len(batch)} / {len(documents)} books stored.")
    time.sleep(5) # Small pause to let the CPU breathe

print("\n✨ ALL BOOKS STORED SUCCESSFULLY!")

🚀 Starting ingestion...
✅ Initial 40 books stored.
✅ Progress: 80 / 2941 books stored.
✅ Progress: 120 / 2941 books stored.
✅ Progress: 160 / 2941 books stored.
✅ Progress: 200 / 2941 books stored.
✅ Progress: 240 / 2941 books stored.
✅ Progress: 280 / 2941 books stored.
✅ Progress: 320 / 2941 books stored.
✅ Progress: 360 / 2941 books stored.
✅ Progress: 400 / 2941 books stored.
✅ Progress: 440 / 2941 books stored.
✅ Progress: 480 / 2941 books stored.
✅ Progress: 520 / 2941 books stored.
✅ Progress: 560 / 2941 books stored.
✅ Progress: 600 / 2941 books stored.
✅ Progress: 640 / 2941 books stored.
✅ Progress: 680 / 2941 books stored.
✅ Progress: 720 / 2941 books stored.
✅ Progress: 760 / 2941 books stored.
✅ Progress: 800 / 2941 books stored.
✅ Progress: 840 / 2941 books stored.
✅ Progress: 880 / 2941 books stored.
✅ Progress: 920 / 2941 books stored.
✅ Progress: 960 / 2941 books stored.
✅ Progress: 1000 / 2941 books stored.
✅ Progress: 1040 / 2941 books stored.
✅ Progress: 1080 / 2941

In [5]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Re-initialize the exact same model wrapper used for building
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

# 2. Path to your saved folder
db_path = "./book_db_final_version"

if os.path.exists(db_path):
    # This LOADS the database from disk
    db_books = Chroma(
        persist_directory=db_path,
        embedding_function=hf_embeddings  # Must use the wrapper here!
    )
    
    # Verify the count
    num_books = len(db_books.get()['ids'])
    print(f"✅ Database loaded successfully with {num_books} books.")
else:
    print(f"❌ Error: Folder '{db_path}' not found. Run the ingestion code first!")

✅ Database loaded successfully with 2941 books.


In [7]:
query = "A book about Roman history"
results = db_books.similarity_search(query, k=10)
print("results: ", results)


results:  [Document(id='9a4510f6-2840-4557-8c5c-b2781ef5827c', metadata={'source': 'tagged_descriptions.text'}, page_content="9780380710829 | The lives of ancient Rome's men--general Gaius Marius and his rival Lucius Cornelius Sulla--unfold amid Republican Rome's struggle in a world of treachery and barbarism\n9780380710843 | The fourth novel of the Masters of Rome series focuses on the women in the life of the Roman emperor Gaius Julius Caesar at the height of his power"), Document(id='04ba75a6-5cb9-4870-9ed7-300c27a4cfc8', metadata={'source': 'tagged_descriptions.text'}, page_content="9780192833006 | Cornelius Tacitus, Rome's greatest historian, was inspired to take up his pen when the assassination of Domitian ended `fifteen years of enforced silence'. Agricola is the biography of his late father-in-law and an account of Roman Britain. Germania gives insight into Rome's most dangerous enemies, the Germans, and is the only surviving specimen from the ancient world of an ethnographic 

In [8]:
# .replace('"', '') removes any double quotes found in that first segment
isbn_str = results[0].page_content.split(" ")[0].replace('"', '').strip()
books[books["isbn13"] == int(isbn_str)]
# print(results[0].metadata)

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
1753,9780380710829,038071082X,The Grass Crown,Colleen McCullough,Fiction,http://books.google.com/books/content?id=Kt3gN...,1992.0,4.29,1104.0,9397.0,The Grass Crown,9780380710829 | The lives of ancient Rome's me...


In [17]:
def retrieve_symmentaic_recommendations(
        query: str,
        top_k: int = 10
    ,
)-> pd.DataFrame:
    recs=db_books.similarity_search(query, k=top_k)
    book_list=[]
    for i in range(len(recs)):
        isbn_str = recs[i].page_content.split(" ")[0].replace('"', '').strip()
        book_info=books[books["isbn13"] == int(isbn_str)]
        book_list.append(book_info)
#    return pd.concat(book_list).reset_index(drop=True)
    return pd.concat(book_list).reset_index(drop=True)
    

In [18]:
results_df=retrieve_symmentaic_recommendations("A book about a past",top_k=10)
results_df

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780061159176,0061159174,The Known World,Edward P. Jones,Fiction,http://books.google.com/books/content?id=NE-vs...,2006.0,3.83,388.0,28962.0,The Known World,9780061159176 | One of the most acclaimed nove...
1,9780300094008,0300094000,Marcel Proust,William C. Carter,Biography & Autobiography,http://books.google.com/books/content?id=NxTaY...,2002.0,4.32,992.0,13.0,Marcel Proust: A Life,9780300094008 | This book is a magisterial acc...
2,9780618057078,0618057072,The Origin of Consciousness in the Breakdown o...,Julian Jaynes,Philosophy,http://books.google.com/books/content?id=6Q0kS...,2000.0,4.24,491.0,3258.0,The Origin of Consciousness in the Breakdown o...,"9780618057078 | At the heart of this classic, ..."
3,9781564780614,1564780619,Letters,John Barth,Fiction,http://books.google.com/books/content?id=m0MZc...,1994.0,3.82,772.0,202.0,Letters: A Novel,"9781564780614 | Basically, [Barth] takes sever..."
4,9780393328622,0393328627,The History of Love: A Novel,Nicole Krauss,Fiction,http://books.google.com/books/content?id=-Il7X...,2006.0,3.92,255.0,107024.0,The History of Love: A Novel,9780393328622 | Sixty years after a book's pub...
5,9781593083779,1593083777,Swann's Way,Marcel Proust;Elizabeth Dalton,Fiction,http://books.google.com/books/content?id=wHUIr...,2005.0,4.14,466.0,18.0,Swann's Way,9781593083779 | Presents the first book of Pro...
6,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 | This book tells the tale of a ...
7,9780811215923,081121592X,Borges and the Eternal Orangutans,Luís Fernando Veríssimo;Margaret Jull Costa,Fiction,http://books.google.com/books/content?id=G5cn9...,2005.0,3.79,135.0,538.0,Borges and the Eternal Orangutans,9780811215923 | Vogelstein is a loner who has ...
8,9781565849655,1565849655,Interesting Times,Eric J. Hobsbawm,Biography & Autobiography,http://books.google.com/books/content?id=EgEMv...,2005.0,4.06,448.0,359.0,Interesting Times: A Twentieth-century Life,9781565849655 | Eric Hobsbawm has been widely ...
9,9780060593247,0060593245,Every Book Its Reader,Nicholas A. Basbanes,Literary Criticism,http://books.google.com/books/content?id=o4K_K...,2006.0,3.96,400.0,300.0,Every Book Its Reader: The Power of the Printe...,9780060593247 | Inspired by a landmark exhibit...


In [19]:
books['categories'].value_counts().reset_index().query('count>50')

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Philosophy,117
6,Religion,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [9]:
# Updated mapping to ensure perfect string matching with the model
category_mapping = {
    'Fiction': "Fiction",
    'Juvenile Fiction': "Fiction",
    'Comics & Graphic Novels': "Fiction",
    'Drama': "Fiction",
    'Poetry': "Fiction",
    'Biography & Autobiography': "NonFiction",
    'History': "NonFiction",
    'Literary Criticism': "NonFiction",
    'Philosophy': "NonFiction",
    'Religion': "NonFiction",
    'Science': "NonFiction",
    'Juvenile Nonfiction': "NonFiction"
}

# Apply the mapping
books['broad_category'] = books['categories'].map(category_mapping)

In [10]:
books[~(books['broad_category'].isna())]
# books['broad_category'].value_counts().reset_index().query('count>50')

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,broad_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 | A NOVEL THAT READERS and criti...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 | A memorable, mesmerizing heroi...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 | Tricked once more by his wily ...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 | Joe and his parents are enjoyi...,Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014 | A Few Weeks After Marrying An ...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279 | ""Cornell Woolrich's novels def...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 | Rescued from the lockers in wh...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 | This book is the story of a yo...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 | This collection of the timeles...,NonFiction


In [11]:
from transformers import pipeline,AutoConfig

# Initialize the lightweight classifier
# model="valhalla/distilbart-mnli-12-3" is the faster version
config = AutoConfig.from_pretrained("valhalla/distilbart-mnli-12-3")
config.tie_word_embeddings = False
fiction_categories = ["Fiction", "NonFiction"]
classifier = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-3",
    device=-1  # This forces the model to run on your CPU
)


Device set to use cpu


In [12]:
sequence= books.loc[books['broad_category'] == "Fiction", "tagged_description"].reset_index(drop=True)[0]

In [13]:
# # from os import pipe
# result=classifier(sequence,fiction_categories)
clean_sequence = sequence.split('|')[-1].strip()

# # 3. Classify the cleaned text
result = classifier(clean_sequence, fiction_categories)

print(f"Label: {result['labels'][0]} with score: {result['scores'][0]:.4f}")

Label: Fiction with score: 0.7222


In [14]:
import numpy as np
max_index=np.argmax(classifier(sequence,fiction_categories)['scores'])
my_label=classifier(sequence,fiction_categories)['labels'][max_index]
my_label

'Fiction'

In [15]:
def generate_predictions(sequence, function_categories):
    prediction = classifier(sequence, function_categories)
    max_index = np.argmax(prediction['scores'])
    max_label = prediction['labels'][max_index]
    return max_label

# def generate_predictions(sequence, categories):
#     # 1. Clean the sequence INSIDE the function
#     # This handles the ISBN/Pipe removal automatically
#     clean_sequence = sequence.split('|')[-1].strip()
    
#     # 2. Run the classifier with a template for better context
#     prediction = classifier(
#         clean_sequence, 
#         categories, 
#         hypothesis_template="This book is {}."
#     )
    
#     # 3. Get the top label
#     max_index = np.argmax(prediction['scores'])
#     return prediction['labels'][max_index]

In [16]:
from  tqdm import tqdm
actual_cate = []
predicted_cate = []

# Filter once to avoid the indexing bug
fiction_subset = books[books["broad_category"] == "Fiction"].reset_index(drop=True)

for i in tqdm(range(0, 300)):
    sequence = fiction_subset["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cate.append(pred)
    actual_cate.append("Fiction")

100%|██████████| 300/300 [04:28<00:00,  1.12it/s]


In [ ]:
# Filter once to avoid the indexing bug
fiction_subset = books[books["broad_category"] == "NonFiction"].reset_index(drop=True)

for i in tqdm(range(0, 300)):
    sequence = fiction_subset["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cate.append(pred)
    actual_cate.append("NonFiction")

100%|██████████| 300/300 [04:44<00:00,  1.05it/s]


In [ ]:
predictions_df=pd.DataFrame({"actual_category": actual_cate, "predicted_category": predicted_cate})

In [19]:
predictions_df

,actual_category,predicted_category
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Fiction
3,Fiction,Fiction
4,Fiction,Fiction
...,...,...
595,NonFiction,NonFiction
596,NonFiction,NonFiction
597,NonFiction,NonFiction
598,NonFiction,NonFiction


In [20]:
predictions_df["correct_prediction"]=np.where(predictions_df["actual_category"]==predictions_df["predicted_category"], 1, 0)    

In [21]:
predictions_df["correct_prediction"].sum()/len(predictions_df)

np.float64(0.825)

In [22]:
isbns=[]
predicted_cats=[]

missing_cats=books.loc[books["broad_category"].isna(), ["isbn13", "tagged_description"]].reset_index(drop=True)

In [23]:
for i in tqdm(range(len(missing_cats))):
    sequence = missing_cats["tagged_description"][i]
    
    pred = generate_predictions(sequence, fiction_categories)
    
    predicted_cats.append(pred)
    isbns.append(missing_cats["isbn13"][i]) 

100%|██████████| 1454/1454 [21:26<00:00,  1.13it/s]


In [24]:
missing_predictions_df=pd.DataFrame({"isbn13": isbns, "predicted_category": predicted_cats})
missing_predictions_df

,isbn13,predicted_category
0,9780002261982,Fiction
1,9780006280897,NonFiction
2,9780006280934,NonFiction
3,9780006380832,NonFiction
4,9780006470229,Fiction
...,...,...
1449,9788125026600,NonFiction
1450,9788171565641,Fiction
1451,9788172235222,Fiction
1452,9788173031014,NonFiction


In [25]:
books=pd.merge(books, missing_predictions_df, on="isbn13", how="left")
books["books_category"]=np.where(books["broad_category"].isna(), books["predicted_category"], books["broad_category"])
books=books.drop(columns=["predicted_category"])

In [26]:
books.to_csv("books_with_categories.csv", index=False)